Revision de los datos

Inicialmente leeremos los datos de nuestro archivo csv y lo guardaremos en dataframe

In [1]:
import pandas as pd

df = pd.read_csv('./data/dataset.csv', index_col='id')
df.head(10)

FileNotFoundError: [Errno 2] No such file or directory: './data/dataset.csv'

Podemos ver que nuestro dataset tiene 49 columnas la cual contiene la variable objetivo en la columna `CLASS_LABEL`

Ahora intentaremos ver la distribucion de las claases, para averiguar si hay desbalance de clases

In [14]:
import numpy as np 

y = df.loc[:,'CLASS_LABEL'].to_numpy()

np.unique_counts(y)

UniqueCountsResult(values=array([0, 1]), counts=array([5000, 5000]))

Vemos que tenemos todos los datos balanceados es decir $10000$ datos con $5000$ para la clase positiva y $5000$ para la clase negativa

Ahora revisaremos si existe datos faltantes dentro de las columnas

In [15]:
df.isnull().sum().sort_values(ascending=False)

NumDots                               0
SubdomainLevel                        0
PathLevel                             0
UrlLength                             0
NumDash                               0
NumDashInHostname                     0
AtSymbol                              0
TildeSymbol                           0
NumUnderscore                         0
NumPercent                            0
NumQueryComponents                    0
NumAmpersand                          0
NumHash                               0
NumNumericChars                       0
NoHttps                               0
RandomString                          0
IpAddress                             0
DomainInSubdomains                    0
DomainInPaths                         0
HttpsInHostname                       0
HostnameLength                        0
PathLength                            0
QueryLength                           0
DoubleSlashInPath                     0
NumSensitiveWords                     0


Según los resultados obtenidos vemos que no existen datos faltantes 

## Verificación de redundancia en los datos

### Matriz de correlacion

Para verificar si existen columnas correlacionadas creamos la matriz de correlaciones y verificamos si hay algun par de columnas con una correlacion mayor o igual al $0.8$

In [16]:
import numpy as np

corr = df.corr(numeric_only=True).abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

pairs = (
    upper.stack()                     
    .reset_index()
    .rename(columns={"level_0": "variable_1", "level_1": "variable_2", 0: "correlacion"})
)

corr_pairs = (
    pairs[pairs["correlacion"] > 0.8]
    .sort_values("correlacion", ascending=False)
    .reset_index(drop=True)
)

corr_pairs

,variable_1,variable_2,correlacion
0,NumQueryComponents,NumAmpersand,0.872951
1,AbnormalFormAction,AbnormalExtFormActionR,0.818794
2,NumQueryComponents,QueryLength,0.811784
3,PctExtResourceUrls,PctExtResourceUrlsRT,0.804744
4,UrlLength,UrlLengthRT,0.800096


Ahora comprobaremos la correlacion de cada una de estas columnas con la variable objetivo

In [17]:
corr_cols = np.unique(corr_pairs.drop(columns='correlacion'))
corr_cols = df[corr_cols]

y = df["CLASS_LABEL"]
corr_target = corr_cols.corrwith(y).abs().sort_values(ascending=False)

corr_target

AbnormalExtFormActionR    0.185799
UrlLengthRT               0.169513
NumQueryComponents        0.147391
AbnormalFormAction        0.145073
NumAmpersand              0.081446
QueryLength               0.080525
UrlLength                 0.074493
PctExtResourceUrlsRT      0.052089
PctExtResourceUrls        0.021922
dtype: float64

- Las variables `NumQueryComponents` y `NumAmpersand` estan muy relacionadas y son informacion redundante pues en las urls las componentes del query son separados por el `&`, y de entre estas dos la que mejor relacion tiene con la variable objetivo es `NumQueryComponents`.
- `AbnormalFormAction` y `AbnormalExtFormActionR` apesar de que tienen una correlacion alta, parece que en el dominio son variables diferentes, la una habla de formularios con acciones anormales y la otra con formularios con enlaces externos asi que ambas se conservan. 	
- Lo mismo pasa entre `NumQueryComponents` y `QueryLength`, si la url tiene muchos componentes por consecuencia la query sera mas extensa, de entre estas dos la que mejor relacion tiene con la variable objetivo otra vez es la variable `NumQueryComponents`
- Las variables `PctExtResourceUrls` y `PctExtResourceUrlsRT`, estan correlacionadas pues representan los mismos datos pero la segunda esta categorizada, asi que nos quedamos con `PctExtResourceUrlsRT` quien tiene mayor correlacion con la variable objetivo
- La variable `UrlLength` esta correlacionada `UrlLengthRT`  y aun asi es la segunda la que esta mas correlacionada con la variable objetivo, pues RT son versiones categorizadas de la misma variable `(-1, 0, 1)` evaluados segun expertos en phishing

Del análisis anterior tenemos las siguientes variables a eliminar por correlacion alta: 
`[NumAmpersand, QueryLength, PctExtResourceUrls, UrlLength]`

In [18]:
cols_to_drop = ['NumAmpersand', 'QueryLength', 'PctExtResourceUrls', 'UrlLength']
df = df.drop(columns=cols_to_drop)
df.shape

(10000, 45)

### Varianza de las variables 

Identificaremos variables que casi no varian o que se mantiene constantes casi todo el tiempo, es decir no aportan mucho

In [19]:
from sklearn.feature_selection import VarianceThreshold

X = df.drop(columns='CLASS_LABEL')
y = df['CLASS_LABEL']\
    
selector = VarianceThreshold(threshold=0.01)
selector.fit(X)
constantes = X.columns[~selector.get_support()]

for col in constantes: 
    print(f"columna: {col} -- {np.unique_counts(df[col])}")

columna: AtSymbol -- UniqueCountsResult(values=array([0, 1]), counts=array([9997,    3]))
columna: NumHash -- UniqueCountsResult(values=array([0, 1]), counts=array([9977,   23]))
columna: HttpsInHostname -- UniqueCountsResult(values=array([0]), counts=array([10000]))
columna: DoubleSlashInPath -- UniqueCountsResult(values=array([0, 1]), counts=array([9991,    9]))
columna: FakeLinkInStatusBar -- UniqueCountsResult(values=array([0, 1]), counts=array([9945,   55]))
columna: PopUpWindow -- UniqueCountsResult(values=array([0, 1]), counts=array([9951,   49]))


In [22]:
rows = []
for col in constantes: 
    rows.append({
        'nombre de la columna': col,
        'totales positivas': df[col].sum(), 
        'media positivas': df[col].mean(),
        'totales positivas que son phishing': df[df[col] == 1]['CLASS_LABEL'].sum(), 
        'media positivas que son phising': df[df[col] == 1]['CLASS_LABEL'].mean()
    })

pd.DataFrame(rows)

,nombre de la columna,totales positivas,media positivas,totales positivas que son phishing,media positivas que son phising
0,AtSymbol,3,0.0003,0,0.000000
1,NumHash,23,0.0023,23,1.000000
2,HttpsInHostname,0,0.0000,0,NaN
3,DoubleSlashInPath,9,0.0009,8,0.888889
4,FakeLinkInStatusBar,55,0.0055,22,0.400000
5,PopUpWindow,49,0.0049,1,0.020408


De primeras vemos que la columna `HttpsInHostname` no tiene variabilidad asi que no aporta nada al modelo asi que la eliminamos

- En la columna `AtSymbol` de las 3 que son positivas ninguna es phishing mostrando una poca relacion con la variable objetivo, asi que la eliminamos
- Lo mismo sucede con `PopUpWindow` solo una de las 49 son positivas para phishing asi que no aporta mucho al modelo, asi que se elimina. 

In [23]:
df = df.drop(columns=["HttpsInHostname", "AtSymbol", 'PopUpWindow'])
df.shape

(10000, 42)

Finalmente guardamos este dataframe dentro de variables ya filtradas en la ruta del proyecto `./data/filtereddataset.csv`

In [36]:
df.to_csv('./data/filtereddataset.csv')